# gpt-oss P2 — direction loudness per layer

Which layer is most direction-loaded, over every reasoning token of the train 2,880, the same
population as the gpt-oss grid profile. On gpt-oss the probe layer is 15 by convention
(force-kept in every tree); this profile describes where that sits, it does not choose it.

Input is what `wrappers/gptoss_analysis/direction/1_loudest_layer/join_loudness_profile.sh`
writes from the REUSED trees (`jlens_mass_l15`, `logitlens_mass_l15`): one row per reasoning
token, one `L{layer}` column per layer (7..23), each cell `log P(any direction word)`. Those
trees hold all 3,600 mass-era trajectories, eval 720 included, so cell 1 keeps only the names in
`/workspace/splits/mass_train_2880.txt`.

An **empty** cell means the lens did not cover that layer — not "no mass here". It becomes
`NaN` and is skipped by the mean.

In [ ]:
# 1. imports and load the joined tables
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

OUT_DIR = Path("/workspace/loudness_evaluation/gptoss_p2_layer_profile")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# keep_default_na=False: decoded tokens include the literal string "NA" and empty
# strings, which pandas would otherwise turn into missing values.
jlens = pd.read_csv(OUT_DIR / "jlens_tokens.csv", keep_default_na=False, low_memory=False)
logitlens = pd.read_csv(OUT_DIR / "logitlens_tokens.csv", keep_default_na=False, low_memory=False)

# The joined trees hold all 3,600 mass-era trajectories. Keep the train 2,880, as the grid profile
# does, so the eval 720 never enters a profile and the two profiles average over the same population.
TRAIN_NAMES = set(Path("/workspace/splits/mass_train_2880.txt").read_text().split())
jlens = jlens[jlens.trajectory.isin(TRAIN_NAMES)].reset_index(drop=True)
logitlens = logitlens[logitlens.trajectory.isin(TRAIN_NAMES)].reset_index(drop=True)
assert jlens.trajectory.nunique() == logitlens.trajectory.nunique() == len(TRAIN_NAMES)

print(f"jlens:     {len(jlens):,} token rows, {jlens.trajectory.nunique():,} trajectories")
print(f"logitlens: {len(logitlens):,} token rows, {logitlens.trajectory.nunique():,} trajectories")
jlens.head()

In [ ]:
# 2. aggregate at row level — every token is one observation
def layer_columns(df):
    """The L{layer} columns, in layer order."""
    cols = [c for c in df.columns if c.startswith("L") and c[1:].isdigit()]
    return sorted(cols, key=lambda c: int(c[1:]))


def loudness_matrix(df):
    """Just the layer columns, numeric. Empty cells (layer not covered) become NaN."""
    cols = layer_columns(df)
    return df[cols].apply(pd.to_numeric, errors="coerce")


jlens_mass = loudness_matrix(jlens)
logitlens_mass = loudness_matrix(logitlens)

print("layers:", [int(c[1:]) for c in layer_columns(jlens)])
print(f"jlens rows: {len(jlens_mass):,}   logitlens rows: {len(logitlens_mass):,}")

In [ ]:
# 3. mean direction loudness per layer
def mean_per_layer(mass):
    """{layer: mean loudness}, as a Series indexed by integer layer."""
    means = mass.mean()  # skips NaN
    means.index = [int(c[1:]) for c in means.index]
    return means.sort_index()


jlens_per_layer = mean_per_layer(jlens_mass)
logitlens_per_layer = mean_per_layer(logitlens_mass)

print(f"jlens argmax layer:     {jlens_per_layer.idxmax()}  ({jlens_per_layer.max():.4f})")
print(f"logitlens argmax layer: {logitlens_per_layer.idxmax()}  ({logitlens_per_layer.max():.4f})")
pd.DataFrame({"jlens": jlens_per_layer, "logitlens": logitlens_per_layer})

In [ ]:
# 4. J-Lens Direction Loudness Per Layer
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(jlens_per_layer.index, jlens_per_layer.values, marker="o")
ax.axvline(jlens_per_layer.idxmax(), color="crimson", linestyle="--", linewidth=1)
ax.set_title("J-Lens Direction Loudness Per Layer")
ax.set_xlabel("layer")
ax.set_ylabel("J-lens loudness")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "jlens_loudness_per_layer.png", dpi=160, bbox_inches="tight")
print("saved", FIG_DIR / "jlens_loudness_per_layer.png")
plt.show()

In [ ]:
# 5. Logitlens Direction Loudness Per Layer
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(logitlens_per_layer.index, logitlens_per_layer.values, marker="o", color="tab:orange")
ax.axvline(logitlens_per_layer.idxmax(), color="crimson", linestyle="--", linewidth=1)
ax.set_title("Logitlens Direction Loudness Per Layer")
ax.set_xlabel("layer")
ax.set_ylabel("Logitlens loudness")
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "logitlens_loudness_per_layer.png", dpi=160, bbox_inches="tight")
print("saved", FIG_DIR / "logitlens_loudness_per_layer.png")
plt.show()